<a href="https://colab.research.google.com/github/essensha/cs2-anti-cheat-ml-research/blob/main/Graph_Based_Coordinated_Cheating_(Nodes_and_Edges).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file_path = "/srv/synthetic_anticheat_telemetry.csv"

# Reload and overwrite the old DataFrame
df = pd.read_csv(file_path, skiprows=1, index_col=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(5))

Shape: (12000, 11)
Columns: ['player_id', 'match_id', 'match_day', 'skill_proxy', 'actions_per_min', 'median_reaction_ms', 'timing_regularities', 'target_accuracy', 'repeated_action_ratio', 'synthetic_pattern', 'is_suspicious_simulated']


,player_id,match_id,match_day,skill_proxy,actions_per_min,median_reaction_ms,timing_regularities,target_accuracy,repeated_action_ratio,synthetic_pattern,is_suspicious_simulated
0,100000,204010,27,0.616,151.3,334.8,0.383,0.510,0.066,normal,0
1,100001,200988,18,0.333,143.9,290.8,0.240,0.366,0.026,normal,0
2,100002,204639,52,0.741,155.3,172.5,0.537,0.613,0.421,normal,0
3,100003,203584,31,0.328,127.1,249.7,0.255,0.309,0.146,normal,0
4,100004,204014,19,0.418,129.4,247.0,0.176,0.460,0.204,normal,0


In [ ]:

import pandas as pd
import networkx as nx

# Synthetic player relationships
edges = pd.DataFrame({
    "player_a": ["A", "A", "B", "D", "E", "F"],
    "player_b": ["B", "C", "C", "E", "F", "G"],
    "games_together": [42, 38, 45, 12, 8, 5]
})

G = nx.from_pandas_edgelist(
    edges,
    source="player_a",
    target="player_b",
    edge_attr="games_together"
)

print("Players:", G.number_of_nodes())
print("Connections:", G.number_of_edges())

for player in G.nodes:
    print(
        player,
        "degree:",
        G.degree(player)
    )


Players: 7
Connections: 6
A degree: 2
B degree: 2
C degree: 2
D degree: 1
E degree: 2
F degree: 2
G degree: 1


Degree counts unique connections, not games played.

In [ ]:
for player in G.nodes:
    print(
        player,
        G.degree(player, weight="games_together")
    )

A 80
B 87
C 83
D 12
E 20
F 13
G 5


Degree: Number of unique player connections.



Weighted degree: Total games played across those connections.

In [ ]:
print(nx.triangles(G))

{'A': 1, 'B': 1, 'C': 1, 'D': 0, 'E': 0, 'F': 0, 'G': 0}


In [ ]:
print("Clustering coefficients:")
for player, score in nx.clustering(G).items():
    print(player, round(score, 2))

Clustering coefficients:
A 1.0
B 1.0
C 1.0
D 0
E 0
F 0
G 0


### Stage 2 — First modeling exercise

In [ ]:
import pandas as pd

risk = pd.DataFrame({
    "player": ["A", "B", "C", "E"],
    "behavior_score": [0.90, 0.85, 0.88, 0.92],
    "clustering": [1.0, 1.0, 1.0, 0.0]
})

# Illustrative ranking score, not calibrated probability
risk["combined_score"] = (
    0.7 * risk["behavior_score"]
    + 0.3 * risk["clustering"]
)

print(risk.sort_values("combined_score", ascending=False))

  player  behavior_score  clustering  combined_score
0      A            0.90         1.0           0.930
2      C            0.88         1.0           0.916
1      B            0.85         1.0           0.895
3      E            0.92         0.0           0.644


For coordinated-cheating detection, compare:
- Baseline: Behavioral features only.
- Candidate: Behavioral + graph features.
- Metric: Precision@K under a fixed investigation budget, using independent label

In [ ]:
for weight in [0.5, 0.6, 0.7, 0.8, 1.0]:
    risk[f"score_{weight}"] = (
        weight * risk["behavior_score"]
        + (1 - weight) * risk["clustering"]
    )

    ranked = risk.sort_values(
        f"score_{weight}",
        ascending=False
    )

    print(f"\nBehavior weight: {weight}")
    print(ranked[["player", f"score_{weight}"]])


Behavior weight: 0.5
  player  score_0.5
0      A      0.950
2      C      0.940
1      B      0.925
3      E      0.460

Behavior weight: 0.6
  player  score_0.6
0      A      0.940
2      C      0.928
1      B      0.910
3      E      0.552

Behavior weight: 0.7
  player  score_0.7
0      A      0.930
2      C      0.916
1      B      0.895
3      E      0.644

Behavior weight: 0.8
  player  score_0.8
0      A      0.920
2      C      0.904
1      B      0.880
3      E      0.736

Behavior weight: 1.0
  player  score_1.0
3      E       0.92
0      A       0.90
2      C       0.88
1      B       0.85


In [ ]:
# Confirmed cheaters among the top 100 investigated
results = {
    "Behavioral only": 65,
    "Behavioral + graph": 82
}

investigation_capacity = 100

for model, confirmed in results.items():
    precision = confirmed / investigation_capacity
    print(f"{model}: Precision@100 = {precision:.2%}")

Behavioral only: Precision@100 = 65.00%
Behavioral + graph: Precision@100 = 82.00%


In [ ]:
import pandas as pd

events = pd.DataFrame({
    "event_id": [101, 102, 102, 103, 103, 103],
    "player_a": ["A", "A", "A", "B", "B", "B"],
    "player_b": ["B", "C", "C", "C", "C", "C"]
})

clean_events = events.drop_duplicates(subset=["event_id"])

print("Before:", len(events))
print("After:", len(clean_events))
print(clean_events)

Before: 6
After: 3
   event_id player_a player_b
0       101        A        B
1       102        A        C
3       103        B        C


### Stage 5: Time-decayed graph features.

In [ ]:
import numpy as np
import pandas as pd

connections = pd.DataFrame({
    "pair": ["A-B", "C-D", "E-F"],
    "games_together": [50, 50, 50],
    "days_since_last_game": [180, 7, 30]
})

# Connection weight halves every 30 days
half_life = 30

connections["decayed_weight"] = (
    connections["games_together"]
    * 0.5 ** (
        connections["days_since_last_game"] / half_life
    )
)

print(connections.round(2))

  pair  games_together  days_since_last_game  decayed_weight
0  A-B              50                   180            0.78
1  C-D              50                     7           42.53
2  E-F              50                    30           25.00
